# Deploying a Masked Language Model

The DistilBERT language model derives from Google BERT language model, which can be used for masked language modeling and next sentence prediction tasks.

### 1. Define a project and a `huggingfaceserve` function 

In [ ]:
import os

import digitalhub as dh

p_name = f"tutorial-project-{os.environ['USER']}"
project = dh.get_or_create_project(p_name)

In [ ]:
llm_function = project.new_function(
    "llm_masked",
    kind="huggingfaceserve",
    model_name="mymodel",
    path="huggingface://distilbert/distilbert-base-uncased",
)

### 2. Serve the model

LLM models have particular hardware requirements. When serving a model within the platform, you can use one of the preconfigured profiles, which define the resources that will be allocated.

**NOTE**: when requesting a GPU node for the service, it may take some time for the service to start.

In [ ]:
llm_run = llm_function.run(action="serve", profile="1xV100", wait=True)

### 3. Try the model

Note that BERT answers reflect its training on English Wikipedia and BookCorpus.

In [ ]:
model_name = "mymodel"
json = {
    "inputs": [
        {
            "name": "input-0",
            "shape": [1],
            "datatype": "BYTES",
            "data": ["Cats are [MASK]."],
        },
    ]
}

llm_run.invoke(model_name=model_name, json=json).json()

# Adapt the Model on Movie Reviews Domain

### 1. Fine-tune the model

Define and run a training function that will create a new model trained on the IMDb dataset.

In [ ]:
train_func = project.new_function(
    name="train_model",
    kind="python",
    python_version="PYTHON3_10",
    code_src="src/functions.py",
    handler="train",
    requirements=[
        "hf_xet",
        "datasets",
        "transformers[torch]",
        "torch",
        "torchvision",
        "accelerate",
    ],
)

In [ ]:
train_func.run(action="build", wait=True)
train_run = train_func.run(action="job", profile="1xV100", wait=True)

### 2. Serve the fine-tuned model

Create and run the serving function (this will create a new version of the function created during the first step).

In [ ]:
model = dh.get_model("test_llm_model", project="llm")

In [ ]:
llm_function = project.new_function(
    "llm_masked",
    kind="huggingfaceserve",
    model_name="test_llm_model",
    path=model.spec.path,
)

**NOTE**: at the time of writing, specifying a volume was a temporary workaround to overcome directory space limitations and might not be necessary anymore.

In [ ]:
llm_run_finetuned = llm_function.run(
    action="serve",
    profile="1xV100",
    resources={"disk": "10Gi"},
    wait=True,
)

### 3. Test the fine-tuned model

In [ ]:
model_name_finetuned = "test_llm_model"
json = {
    "inputs": [
        {
            "name": "input-0",
            "shape": [1],
            "datatype": "BYTES",
            "data": ["This [MASK] was great."],
        }
    ]
}

llm_run_finetuned.invoke(model_name=model_name_finetuned, json=json).json()

### 4. Create a Streamlit app

Write the model name and the run key in an environment file that will be accessible by the Streamlit app.

In [ ]:
with open(".env", "w") as f:
    f.write(f"model_name={model_name_finetuned}\n")
    f.write(f"model_run_key={llm_run_finetuned.key}")

In [ ]:
%pip install streamlit dotenv

In [ ]:
! streamlit run src/app.py

If you are running this notebook inside a Coder workspace, navigate to your workspace and click on "Open ports" to find a link to the Streamlit app.